# **Raw functions**

In [11]:
import cv2
import numpy as np

aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
detector = cv2.aruco.ArucoDetector(aruco_dict, cv2.aruco.DetectorParameters())

def detect_aruco(img):
    corners, ids, _ = detector.detectMarkers(img)
    results = []
    if ids is None:
        return results

    for marker_id, c in zip(ids.flatten(), corners):
        c = c.reshape(4, 2)                       
        center = c.mean(axis=0)
        front = (c[0] + c[1]) / 2                 
        back = (c[2] + c[3]) / 2                  
        theta = np.arctan2(front[1] - back[1], front[0] - back[0])
        results.append((int(marker_id), (int(center[0]), int(center[1])), np.degrees(theta), c))
    return results

def aruco_corners(img, ids=(0, 1, 2, 3)):
    corners, found_ids, _ = detector.detectMarkers(img)
    markers = {}
    if found_ids is not None:
        markers = {int(i): c.reshape(4, 2) for i, c in zip(found_ids.flatten(), corners)}
    if not all(i in markers for i in ids):
        return None, markers
    pts = np.array([markers[i].mean(axis=0) for i in ids], dtype="float32")
    return pts, markers

def warp_from_points(img, pts, size=None):
    tl, tr, br, bl = pts
    if size is None:
        width = int(max(np.linalg.norm(tr - tl), np.linalg.norm(br - bl)))
        height = int(max(np.linalg.norm(bl - tl), np.linalg.norm(br - tr)))
    else:
        width, height = size
    dst = np.array([[0, 0], [width - 1, 0],
                    [width - 1, height - 1], [0, height - 1]], dtype="float32")
    M = cv2.getPerspectiveTransform(pts, dst)
    warped = cv2.warpPerspective(img, M, (width, height))
    
    return warped
    
def detect_walls(img):
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    #thresh = cv2.adaptiveThreshold(img_gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY_INV, 81, 31)
    _, thresh = cv2.threshold(img_gray, 60, 255, cv2.THRESH_BINARY_INV)
    
    analysis = cv2.connectedComponentsWithStats(thresh, 4, cv2.CV_32S)
    (totalLabels, label_ids, values, centroid) = analysis
    
    walls = np.zeros(thresh.shape, dtype="uint8")
    min_area = 850
    
    for i in range(1, totalLabels):
        area = values[i, cv2.CC_STAT_AREA]  

        if (area > min_area) :
            componentMask = (label_ids == i).astype("uint8") * 255
            walls = cv2.bitwise_or(walls, componentMask)
            
    return walls
    
def add_safety_margin(grid, margin=3):
    wall_mask = (grid == 1).astype(np.uint8)

    size = 2*margin + 1
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (size, size))
    dilated = cv2.dilate(wall_mask, kernel)

    grid[(dilated == 1) & (grid == 0)] = 4
    return grid

def to_grid_walls(walls, margin, rows, cols):
    grid = overlay = np.zeros((rows, cols), dtype=np.uint8)
    h, w = walls.shape
    
    for r in range(rows):
        for c in range(cols):
            y_start = int(r*h/rows)
            y_end   = int((r + 1)*h/rows)
            x_start = int(c*w/cols)
            x_end   = int((c + 1)*w/cols)
            
            cell = walls[y_start:y_end, x_start:x_end]
            if np.mean(cell) > 100:
                grid[r, c] = 1 
            else:
                grid[r, c] = 0

    grid = add_safety_margin(grid, margin=margin)
    
    return grid

def to_grid_pos(grid, thymio_coord, finish_coord, rows, cols, h, w):
    if thymio_coord != None:
        x_thymio, y_thymio = thymio_coord[0]
        
        c = int(x_thymio*cols/w)
        r = int(y_thymio*rows/h)
        c_end = int(c + 5*np.cos(thymio_coord[1]))
        r_end = int(r + 5*np.sin(thymio_coord[1]))
        
        grid[r, c] = 2

    if finish_coord != None:
        x_finish, y_finish = finish_coord
        r = int(y_finish*rows/h)
        c = int(x_finish*cols/w)
        grid[r, c] = 3
    
    return grid

def draw_grid(grid):
    rows, cols = grid.shape
    img = np.full((rows, cols, 3), 255, dtype=np.uint8)   # white background (BGR)

    img[grid == 1] = (0, 0, 0)
    img[grid == 2] = (0, 200, 0)
    img[grid == 3] = (0, 0, 255)
    img[grid == 4] = (180, 180, 180)

    return img
    
def find_light(mask, min_area=20):
    contour, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contour:
        return None

    largest = max(contour, key=cv2.contourArea)
    
    if cv2.contourArea(largest) < min_area:
        return None
    
    M = cv2.moments(largest)

    if M['m00'] == 0:
        return None
    
    cx = int(M['m10']/M['m00'])
    cy = int(M['m01']/M['m00'])

    return cx, cy

def detect_finish(img):
    img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    
    lower1 = np.array([0, 120, 70])
    upper1 = np.array([10, 255, 255])

    lower2 = np.array([170, 120, 70])
    upper2 = np.array([179, 255, 255])

    mask1 = cv2.inRange(img_hsv, lower1, upper1)
    mask2 = cv2.inRange(img_hsv, lower2, upper2)
    mask = cv2.bitwise_or(mask1, mask2)

    kernel = np.ones((5, 5), np.uint8)

    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    finish_coord = find_light(mask)

    if finish_coord == None:
        return None

    return finish_coord

def robot_pose(warped, robot_id=4):
    corners, ids, _ = detector.detectMarkers(warped)
    if ids is None:
        return None
    for marker_id, c in zip(ids.flatten(), corners):
        if int(marker_id) == robot_id:
            c = c.reshape(4, 2)
            center = c.mean(axis=0)
            front = (c[0] + c[1]) / 2
            back = (c[2] + c[3]) / 2
            theta = np.arctan2(front[1] - back[1], front[0] - back[0])
            return (int(center[0]), int(center[1])), theta
    return None

def draw_direction(img, coordinates):
    if coordinates == None :
        return img, None

    center, theta = coordinates
    
    img_full = cv2.circle(img, center, 5, (0, 0, 255), -1)

    x_end = int(center[0] + 100*np.cos(theta))
    y_end = int(center[1] + 100*np.sin(theta))

    img_full = cv2.arrowedLine(img_full, center, (x_end, y_end), (100, 100, 0), 5)

    sz = img.shape
    h, w = sz[0], sz[1]
    
    overlay = np.zeros((h, w, 4), dtype=np.uint8)
    cv2.circle(overlay, center, 5, (0, 0, 255, 255), -1)
    cv2.arrowedLine(overlay, center, (x_end, y_end), (100, 100, 0, 255), 5)
    
    return img_full, overlay

def draw_finish(img, coordinates):
    if coordinates == None:
        return None
    
    sz = img.shape
    h, w = sz[0], sz[1]
    
    overlay = np.zeros((h, w, 4), dtype=np.uint8)
    cv2.circle(overlay, coordinates, 5, (0, 0, 255, 255), -1)

    return overlay

def merge(base_bgr, overlay_bgra):
    overlay_bgra = cv2.resize(overlay_bgra, (base_bgr.shape[1], base_bgr.shape[0]), interpolation=cv2.INTER_NEAREST)
    alpha = overlay_bgra[:, :, 3:4] / 255.0 
    out = base_bgr * (1 - alpha) + overlay_bgra[:, :, :3] * alpha
    return out.astype(np.uint8)

def capture_obstacles(sec, cap, rows, cols):
    for i in range(sec*10):
        ok, frame = cap.read()
        _, jpeg = cv2.imencode('.jpg', frame)
        ecran.value = jpeg.tobytes()
        
    ok, frame = cap.read()
    
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gauss_frame = cv2.GaussianBlur(gray_frame, (5, 5), 0)
    filtered_frame = cv2.Canny(gauss_frame, 50, 150)
    contours_frame, hierarchy = cv2.findContours(filtered_frame, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours_frame:
        largest = max(contours_frame, key=cv2.contourArea)
        epsilon = 0.02 * cv2.arcLength(largest, True)
        approx = cv2.approxPolyDP(largest, epsilon, True)

        if len(approx) == 4:
            pts = approx.reshape(4, 2)
    
            warped = four_point_transform(frame, pts.astype("float32"))
            walls = detect_walls(warped)

            h, w = walls.shape
            
            grid = to_grid_walls(walls, 15, rows, cols)
    
    return grid, h, w

def fit_height(img, height):
    scale = height / img.shape[0]
    return cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)

# **Run code**

**Run code**

In [4]:
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display

In [12]:
cap = cv2.VideoCapture(1, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
print("camera opened:", cap.isOpened())

ecran = widgets.Image(format="jpeg")
display(ecran)

rows, cols = 100, 100
obstacles_grid = None
thymio_coord = None
finish_coord = None

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            print("cap.read() failed, stopping")
            break

        pts, markers = aruco_corners(frame)
        pts_ok = None
        
        if pts is not None:
            pts_ok = pts

        if pts_ok is None:                       # corners not found yet: show the live camera
            _, jpeg = cv2.imencode('.jpg', frame)
            ecran.value = jpeg.tobytes()
            continue

        warped = warp_from_points(frame, pts_ok)
        h, w = warped.shape[:2]
    
        obstacles_grid = to_grid_walls(detect_walls(warped), 3, rows, cols)

        pose = robot_pose(warped)                # detect before drawing on warped
        if pose is not None:
            thymio_coord = pose
        finish_coord = detect_finish(warped)

        grid_full = to_grid_pos(obstacles_grid.copy(), thymio_coord, finish_coord, rows, cols, h, w)
        grid_img = cv2.resize(draw_grid(grid_full), (w, h), interpolation=cv2.INTER_NEAREST)

        warped_w_dir, _ = draw_direction(warped, thymio_coord)
        both = np.hstack([warped_w_dir, grid_img])

        _, jpeg = cv2.imencode('.jpg', both)
        ecran.value = jpeg.tobytes()

finally:
    cap.release()

camera opened: True


Image(value=b'', format='jpeg')

KeyboardInterrupt: 